In [ ]:
# this script reads out the quarterly files supplied by Earth (sometimes need to be modified for Netease and Apple Music China) and consolidates them into one csv file

import pandas as pd
import os

# Inputs (to be changed for every quarter)
quarter = "2025 Q4"
assess_original_files = 0  # if "1" then look in the folder "as sent by Earth" - sometimes, the data for Apple Music China and Netease needs to be adjusted so then the original files need to be modified


if assess_original_files == 1:
    base_dir = f'../../50 KM Group/Royalties/Statements/Karen/Earth/{quarter}/as sent by Earth/'
else:
    base_dir = f'../../50 KM Group/Royalties/Statements/Karen/Earth/{quarter}/'

fx_file = f'../../50 KM Group/Royalties/Statements/Karen/Earth/{quarter}/lookup_fx.csv'

outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'
outputfilename = f"Earth_{quarter}_raw_combined.csv"

path = os.path.join(outputdirectory, outputfilename)


def align_columns(df1, df2):
    missing_in_df1 = df2.columns.difference(df1.columns)
    missing_in_df2 = df1.columns.difference(df2.columns)    
    
    for col in missing_in_df1:
        df1[col] = pd.NA
    for col in missing_in_df2:
        df2[col] = pd.NA
        
    return df1, df2

def show_data(df):
    if not df.empty:
        print(f"Total fee: {df['Royalties'].sum()}. Total units: {df['Units'].sum()}")
    else:
        print("empty dataframe")

def read_each_sheet(period,file1,file2,content):

    excel_data = pd.ExcelFile(file1)
    df_fx = pd.read_csv(file2)

    df_netease = pd.DataFrame()
    df_apple_china = pd.DataFrame()
    df_万声 = pd.DataFrame()
    df_2022冬奥会 = pd.DataFrame()
    df_TME汇总 = pd.DataFrame()
    df_汽水 = pd.DataFrame()
    df_番茄 = pd.DataFrame()
    df_apple = pd.DataFrame()
    df_spotify = pd.DataFrame()
    df_believe = pd.DataFrame()
    df_linemusic = pd.DataFrame()
    df_taiwan = pd.DataFrame()
    df_kkbox = pd.DataFrame()
    df_网易平台 = pd.DataFrame()
    df_网易订购 = pd.DataFrame()
    df_网易付费单曲 = pd.DataFrame()
    df_JOOX = pd.DataFrame()
    


    # Initialize an empty DataFrame for df_believe
    df_believe = pd.DataFrame()

    # List of platforms for df_believe
    platforms_believe = ["Youtube", "Facebook", "Tik", "Deezer", "Qobuz", "UMA", "Tidal", 
                        "Amazon", "JioS", "Huawei", "Pandora", "YG", "TDC", "Line Music Japan", 
                        "Yandex", "Soundclo", "AWA", "Anghami", "TREBEL", "Fluxus", "Sber",
                        "TikTok","Boom","CapCut","iHeart","NCT"]

    # Dictionary to store DataFrames
    #dataframes = {}


    万声_column_name_mapping = {
        '日期' : 'Period',
        '结算平台' : 'Platform',
        '业务类型' : 'Product',
        '零售价' : 'Price',
        '歌曲ISRC' : 'ISRC',
        '专辑名' : 'Album',
        '歌曲名' : 'Song',
        '歌手名' : 'Artist',
        '录音份额' : 'Share Master Owner',
        '词份额' : 'Share Lyricist',
        '曲份额' : 'Share composer',
        '是否付费歌曲' : 'Paid/not paid',
        '原始版权' : 'Label',
        '结算开始时间' : 'Period start',
        '结算结束时间' : 'Period end',
        '歌曲ID' : 'Song ID',
        '歌曲ISRC' : 'ISRC',
        '专辑UPC' : 'UPC',
        'CP播放次数' : 'Units',
        'CP分成收入CNY' : 'Royalties',
        '原始版权' : 'Label',
    }

    汽水_column_name_mapping = {
        '开始时间' : 'Period start',
        '结束时间' : 'Period end',
        'songID' : 'Song ID',
        '歌名' : 'Song',
        '专辑名' : 'Album',
        '歌手名' : 'Artist',
        'ISRC' : 'ISRC',
        '版权方唯一码' : 'Copyright holder unique code',
        'UPC' : 'UPC',
        '平台' : 'Platform',
        '地区' : 'Country',
        '词比例' : 'Share Lyricist',
        '曲比例' : 'Share composer',
        '结算类型' : 'Settlement type',
        '免费播放量' : 'Free playback volume',
        '免费下载量' : 'Free download volume',
        '付费播放量' : 'Paid playback volume',
        '付费下载量' : 'Paid download volume',
        '单曲订购量' : 'Single order volume',
        '数字专辑订购量' : 'Digital album order volume',
        'MV播放量' : 'MV playback volume',
        '付费音乐服务收入' : 'Paid music service income',
        '广告收入' : 'Advertising income',
        '单曲订购收入' : 'Single order income',
        '数字专辑收入' : 'Digital album income',
        'MV收入' : 'MV income',
        '实际分成收入（CNY）' : 'Royalties',
        '厂牌' : 'Label'
    }

    番茄_column_name_mapping = {
        '开始时间' : 'Period start',
        '结束时间' : 'Period end',
        'songID' : 'Song ID',
        '歌名' : 'Song',
        '专辑名' : 'Album',
        '歌手名' : 'Artist',
        'ISRC' : 'ISRC',
        '版权方唯一码' : 'Copyright holder unique code',
        'UPC' : 'UPC',
        '地区' : 'Country',
        '平台' : 'Platform',
        'saleUnit' : 'Units',
        'lyrics_percentage' : 'Share Lyricist',
        'composition_percentage' : 'Share composer',
        '实际分成收入（CNY）' : 'Royalties',
        '厂牌' : 'Label', 
    }

    apple_column_name_mapping = {
        'Start Date' : 'Period',
        '日期' : 'Period',
        'Platform' : 'Platform',
        'Storefront Name' : 'Country',
        'Country Of Sale' : 'Country',
        'Label/Studio/Network' : 'Label',
        'Label/Studio/Network/Developer/Publisher' : 'Label',
        '币种' : 'Currency',
        'Artist' : 'Artist',
        'Artist/Show/Developer/Author' : 'Artist',
        'Content Title' : 'Song',
        'Title' : 'Song',
        'ISRC' : 'ISRC',
        'ISRC/ISBN' : 'ISRC',
        'Total  Royalty Bearing Plays' : 'Units',
        'Quantity' : 'Units',
        '税前金额' : 'Gross Amount',
        '税前' : 'Gross Amount',
        'USD税前' : 'Gross Amount',
        'Net Royalty Total USD税前' : 'Gross Amount',
        'Net Royalty Total 税前' : 'Gross Amount',
        '预扣税' : 'Withholding Tax',
        '税后金额' : 'Royalties',
        'USD税后' : 'Royalties',
        'Net Royalty Total 税后' : 'Royalties',
        'Net Royalty Total USD税后' : 'Royalties',
        '税后': 'Royalties',
        'Product' : 'Product',
        'Product Type Identifier' : 'Product Type Identifier',
        'Sales or Return' : 'Sales or Return'
    }


    spotify_column_name_mapping = {
        '日期' : 'Period',
        'Country' : 'Country',
        'Product' : 'Product',
        'Label' : 'Label',
        'ISRC' : 'ISRC',
        'Track Name' : 'Song',
        'Artist Name' : 'Artist',
        'Album Name' : 'Album',
        'Noise Content' : 'Noise Content',
        'Quantity' : 'Units',
        'Payable EUR' : 'Royalties',
        '报告的月份' : 'Period',
        '国家 / 地区' : 'Country',
        '流式订阅类型' : 'Streaming Subscription Type',
        '品牌名称' : 'Label',
        'ISRC' : 'ISRC',
        '曲目标题' : 'Song',
        '艺术家姓名' : 'Artist',
        '发行标题' : 'Album',
        'Noise Content' : 'Noise Content',
        '数量' : 'Units',
        '总收入 EUR' : 'Royalties',
        '流媒体订阅类别' : 'Streaming Subscription Category'
    }


    believe_column_name_mapping = {
        '销售月' : 'Period',
        '平台' : 'Platform',
        '国家 / 地区' : 'Country',
        'ISRC' : 'ISRC',
        '客户付款货币' : 'Currency',
        '品牌名称' : 'Label',
        '艺术家姓名' : 'Artist',
        '发行标题' : 'Album',
        '曲目标题' : 'Song',
        'UPC' : 'UPC',
        '发行类型' : 'Issuance',
        '销售类型' : 'Product',
        '数量' : 'Units',
        '总收入 EUR' : 'Royalties',
        '总收入' : 'Royalties',
        '流式订阅类型' : 'Streaming type',
        '流媒体订阅类别' : 'Streaming category',
        '单价' : 'Unit Price'
    }

    linemusic_column_name_mapping = {
        '销售期间 ' : 'Period',
        '服务类型' : 'Product',
        '专辑' : 'Album',
        '歌名' : 'Song',
        '歌手' : 'Artist',
        '歌曲长度' : 'Song Length',
        '次数' : 'Units',
        '未税金额  TWD' : 'Royalties',
        '金额 TWD' : 'Royalties',
        '未税金额 TWD' : 'Royalties',
        'ISRC' : 'ISRC',
        '厂牌名称' : 'Label'
    }

    taiwan_column_name_mapping = {
        '销售期间 ' : 'Period',
        '服务类型' : 'Product',
        '专辑' : 'Album',
        '歌名' : 'Song',
        '歌手' : 'Artist',
        '歌曲长度' : 'Song Length',
        '次数' : 'Units', 
        '国家 / 地区' : 'Country', 
        '未税金额 TWD' : 'Royalties',
        'ISRC' : 'ISRC',
        '厂牌名称' : 'Label'
    }
    
    kkbox_column_name_mapping = {
        '日期' : 'Period',
        'ISRC' : 'ISRC',
        '原厂牌 Label Name' : 'Label',
        '原厂牌名称' : 'Label',
        '艺人名称 Artist Name' : 'Artist',
        '专辑名称 Album Name' : 'Album',
        '歌曲名称 Track Title' : 'Song',
        '歌曲点播次数 Track Units' : 'Units',
        '单曲含税金额TWD' : 'Royalties',
        '单曲金额 TWD' : 'Royalties',
        '地区' : 'Country',
        '税后金额TWD' : 'Royalties',
        '区域' : 'Country'
    }

    网易_columns_name_mapping = {
        '日期' : 'Period',
        'ISRC' : 'ISRC',
        '歌曲名' : 'Song',
        '专辑' : 'Album',
        '艺人' : 'Artist',
        '厂牌名称' : 'Label',
        '付费类型' : 'Play Type',
        '非激励广告-总播放量' : 'Non-incentivized Ads - Total Plays',
        '激励广告-总播放量' : 'Incentivized Ads - Total Plays',
        '总播放量' : 'Units',
        '非激励广告-总下载量' : 'Non-incentivized Ads - Total Downloads',
        '总下载量' : 'Total Downloads',
        '录音授权比例' : 'Share Master Owner',
        '词授权比例' : 'Share Lyricist',
        '曲授权比例' : 'Share composer',
        '表演者授权比例' : 'Share Performer',
        '非激励广告-本月分成收益费用' : 'Non-incentivized Ads - Monthly Share of Revenue',
        '激励广告-本月分成收益费用' : 'Incentivized Ads - Monthly Share of Revenue',
        '本月分成收益费用' : 'Royalties',
        '销售单价' : 'Sales price',
        '销售数量' : 'Units',
        '实际销售收入' : 'Revenue',
        '本月实际销售收益费用' : 'Royalties',
        '本月实际分成收益费用' : 'Royalties',
        '本月分成收益费用（CNY）' : 'Royalties'
    }    
    
    JOOX_columns_name_mapping = {
        '销售月' : 'Period',
        '平台' : 'Platform',
        '国家 / 地区' : 'Country',
        'ISRC' : 'ISRC',
        '品牌名称' : 'Label',
        '艺术家姓名' : 'Artist',
        '发行标题' : 'Album',
        '曲目标题' : 'Song',
        'UPC' : 'UPC',
        '销售类型' : 'Sales Type',
        '数量' : 'Units',
        '客户付款货币' : 'Currency',
        '流式订阅类型' : 'Streaming Subscription Type',
        '流媒体订阅类别' : 'Streaming Subscription Category',
        '总收入' : 'Royalties'
    }

    # Iterate through each sheet in the Excel file
    for sheet_name in excel_data.sheet_names:
        df = excel_data.parse(sheet_name)
        print(f"Reading sheet: {sheet_name}, Rows: {df.shape[0]}, Columns: {df.shape[1]}")

        if "万声" in sheet_name:
            df = df.drop(columns=['结算地区','是否付费歌曲','CP分成收入TWD'],errors = 'ignore')
            df = df.rename(columns=万声_column_name_mapping)
            df["Currency"] = "CNY"
            df["Country"] = "CN"
            df["period_dt"] = pd.to_datetime(df["Period start"], format="%Y/%m/%d")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            #dataframes[sheet_name] = df
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_万声 = df
            
        # now comes the read-in for the CAVCA statements. The sheet tab needs to be adjusted every single time    
        # elif "2022冬奥会" in sheet_name:
        elif "音集协2024" in sheet_name:
            df = df.drop(columns=['结算地区','是否付费歌曲','CP分成收入TWD'],errors = 'ignore')
            df = df.rename(columns=万声_column_name_mapping)
            df["Currency"] = "CNY"
            df["Country"] = "CN"
            df["Sales Quarter"] = "2024 Q1"
            #dataframes[sheet_name] = df
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_2022冬奥会 = df

        elif "TME汇总" in sheet_name:
            df = df.drop(columns=['结算地区','是否付费歌曲','CP分成收入TWD'],errors = 'ignore')
            df = df.rename(columns=万声_column_name_mapping)
            df["Currency"] = "CNY"
            df["Country"] = "CN"
            #dataframes[sheet_name] = df
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_TME汇总 = df
            
        elif "汽水" in sheet_name:
            df = df.rename(columns=汽水_column_name_mapping)
            #dataframes[sheet_name] = df
            df["Currency"] = "CNY"
 #           print(f"Units: {df['Free playback volume'].sum()}")
 #           print(f"Units: {df['Free download volume'].sum()}")
 #           print(f"Units: {df['Paid playback volume'].sum()}")
#            df["Units"] = df["Free playback volume"] + df["Free download volume"] + df["Paid playback volume"] + df["Paid download volume"] + df["Single order volume"] + df["Digital album order volume"] + df["MV playback volume"]
 #           print(f"Units: {df['Units'].sum()}")
            cols = [
                "Free playback volume",
                "Free download volume",
                "Paid playback volume",
                "Paid download volume",
                "Single order volume",
                "Digital album order volume",
                "MV playback volume"
            ]
            df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
            df["Units"] = df[cols].sum(axis=1)
            df = df.drop(columns=['Free playback volume','Free download volume','Paid playback volume','Paid download volume','Single order volume','Digital album order volume','MV playback volume'],errors = 'ignore')
            df = df.drop(columns=['Paid music service income','Advertising income','Single order income','Digital album income','MV income'],errors = 'ignore')
            df = df.drop(columns=['实际分成收入(TWD)'],errors = 'ignore')
            df['Royalties'] = pd.to_numeric(df['Royalties'], errors='coerce')
            df['Units'] = pd.to_numeric(df['Units'], errors='coerce')
            df["period_dt"] = pd.to_datetime(df["Period start"], format="%Y%m%d")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_汽水 = df

        elif "番茄" in sheet_name:
            df = df.rename(columns=番茄_column_name_mapping)
            #dataframes[sheet_name] = df
            df["Currency"] = "CNY"  
            df["period_dt"] = pd.to_datetime(df["Period start"], format="%Y %m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_番茄 = df

        elif "iTunes" in sheet_name:
            df = df.rename(columns=apple_column_name_mapping)
            #df["Currency"] = "USD"
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            #dataframes[sheet_name] = df
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_apple = df
        
        elif "Apple Music China" in sheet_name:
            df = df.rename(columns=apple_column_name_mapping)
            #dataframes[sheet_name] = df
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y %m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_apple_china = df

        elif "网易自洽" in sheet_name:
            df = df.rename(columns=apple_column_name_mapping)
            df = df.drop(columns=['Share'],errors = 'ignore')
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y %m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            df["Currency"] = "CNY"
            #dataframes[sheet_name] = df
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_netease = df

        elif "网易-平台" in sheet_name:
            df = df.rename(columns=网易_columns_name_mapping)
            #df = df.drop(columns=['Share'],errors = 'ignore')
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            df["Currency"] = "CNY"
            df["Platform"] = "Netease"
            #dataframes[sheet_name] = df
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_网易平台 = df

        elif "网易-订购" in sheet_name:
            df = df.rename(columns=网易_columns_name_mapping)
            #df = df.drop(columns=['Share'],errors = 'ignore')
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            df["Currency"] = "CNY"
            df["Platform"] = "Netease"
            #dataframes[sheet_name] = df
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_网易订购 = df

        elif "网易-付费单曲" in sheet_name:
            df = df.rename(columns=网易_columns_name_mapping)
            #df = df.drop(columns=['Share'],errors = 'ignore')
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            df["Currency"] = "CNY"
            df["Platform"] = "Netease"
            #dataframes[sheet_name] = df
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_网易付费单曲 = df

        elif "Spotify" in sheet_name:
            df = df.drop(columns=['Payable CNY'], errors = 'ignore')
            df = df.rename(columns=spotify_column_name_mapping)
            df["Currency"] = "EUR"
            df["Platform"] = "Spotify"
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            #dataframes[sheet_name] = df
            df_spotify = df

        elif "JOOX" in sheet_name:
            df = df.rename(columns=JOOX_columns_name_mapping)
            df["Currency"] = "EUR"
            df["Platform"] = "JOOX"
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            df = df.drop(columns=['机械费'], errors = 'ignore')
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            #dataframes[sheet_name] = df
            df_JOOX = df

        elif any(platform in sheet_name for platform in platforms_believe):
            df = df.drop(columns=['报告的月份'], errors = 'ignore')
            df = df.drop(columns=['平台歌码'], errors = 'ignore')
            df = df.rename(columns=believe_column_name_mapping)
            df["Platform"] = sheet_name
 #           df["Currency"] = "EUR"
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            df_believe = pd.concat([df_believe, df], ignore_index=True)    
            
        elif "Linemusic" in sheet_name:
            df = df.drop(columns=['未税金额  CNY'], errors = 'ignore')
            df = df.drop(columns=['未税金额 CNY'], errors = 'ignore')
            df = df.drop(columns=['平台名称','平台歌码'], errors = 'ignore')
            df = df.rename(columns=linemusic_column_name_mapping)
            df["Platform"] = sheet_name
            df["Currency"] = "NTD"
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            df['Royalties'] = pd.to_numeric(df['Royalties'], errors='coerce')
            df['Units'] = pd.to_numeric(df['Units'], errors='coerce')
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            #dataframes[sheet_name] = df
            df_linemusic = df

        elif "Taiwan" in sheet_name:
            df = df.drop(columns=['未税金额  CNY','未税金额 CNY'], errors = 'ignore')
            df = df.drop(columns=['单曲金额 CNY'], errors = 'ignore')
            df = df.drop(columns=['平台名称','平台歌码','报告的月份'], errors = 'ignore')
            df = df.rename(columns=taiwan_column_name_mapping)
            df["Platform"] = sheet_name
            df["Currency"] = "NTD"
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            #dataframes[sheet_name] = df
            df_taiwan = df
            
        elif "KKBOX" in sheet_name:
            df = df.drop(columns=['单曲含税金额CNY'], errors = 'ignore')
            df = df.drop(columns=['单曲金额 CNY'], errors = 'ignore')
            df = df.drop(columns=['歌曲编号 Song ID'], errors = 'ignore')
            df = df.drop(columns=['歌曲编号'], errors = 'ignore')
            df = df.drop(columns=['单曲金额（原始币种）'], errors = 'ignore')
            df = df.rename(columns=kkbox_column_name_mapping)
            df["Platform"] = sheet_name
            df["Currency"] = "NTD"
            df["period_dt"] = pd.to_datetime(df["Period"], format="%Y%m")
            df["Sales Month"] = df["period_dt"].dt.strftime("%Y%m")
            df["Sales Quarter"] = df["period_dt"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
            print(f"Royalties (loc currency): {df['Royalties'].sum()}. Units: {df['Units'].sum()}")
            #dataframes[sheet_name] = df
            df_kkbox = df

        # After processing all sheets, finalize df_believe
    if not df_believe.empty:
        df_believe["Currency"] = "EUR"
        #dataframes["df_believe"] = df_believe
        print(f"df_believe: Rows: {df_believe.shape[0]}, Columns: {df_believe.shape[1]}")

    # Align columns and concatenate dataframes
    dataframes = [df_apple_china, df_netease, df_JOOX, df_网易平台, df_网易订购, df_网易付费单曲, df_汽水, 
                  df_番茄, df_万声, df_2022冬奥会, df_TME汇总, df_apple, df_spotify, 
                  df_believe, df_linemusic, df_kkbox,df_taiwan]
    df_final = dataframes[0]

    for df in dataframes[1:]:
        #show_data(df)
        df_final, df = align_columns(df_final, df)
        df_final = pd.concat([df_final, df], ignore_index=True)

    #show_data(df_final)

    print(f"All DataFrames concatenated: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")

    # Merge the concatenated DataFrame with df_fx on a common column (replace 'CommonColumn' with the actual column name)
    df_final = pd.merge(df_final, df_fx, on="Currency", how="left")
    df_final["Royalties (USD)"]= df_final["Royalties"]/df_final["FX Rate"]
    df_final["Content"]=content
    print(f"Merged DataFrame: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")

    print(f"Royalties in USD: {df_final['Royalties (USD)'].sum()}. Total units: {df_final['Units'].sum()}")
    #df_final.to_csv(path, index=False)

    return df_final






In [5]:

print(f'Accessing folder: {base_dir}')
df_list = []
for files in os.listdir(base_dir):
    file_string = base_dir + files
    if '曲库' in files:
        print("\nBuilding Back Catalogue")
        df_back_catalogue = read_each_sheet(quarter,file_string, fx_file,"曲库")
    elif '扶摇' in files:
        print("\nBuilding Fu Yao")
        df_fuyao = read_each_sheet(quarter,file_string, fx_file,"扶摇")
    elif '自洽' in files:
        print("\nBuilding Zi Qia")
        df_ziqia = read_each_sheet(quarter,file_string, fx_file,"自洽")
    else:    
        print(f"\n{file_string} UNKNOWN FILE")



Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Earth/2025 Q4/

../../50 KM Group/Royalties/Statements/Karen/Earth/2025 Q4/.DS_Store UNKNOWN FILE

../../50 KM Group/Royalties/Statements/Karen/Earth/2025 Q4/Earth_2025 Q4_raw_combined.csv UNKNOWN FILE

../../50 KM Group/Royalties/Statements/Karen/Earth/2025 Q4/Earth_2025 Q4_raw_combined_old.csv UNKNOWN FILE

../../50 KM Group/Royalties/Statements/Karen/Earth/2025 Q4/lookup_fx.csv UNKNOWN FILE

Building Zi Qia
Reading sheet: 总表, Rows: 46, Columns: 16
Reading sheet: Apple Music China, Rows: 6, Columns: 14
Royalties (loc currency): 438.64. Units: 39619
Reading sheet: 网易自洽, Rows: 24, Columns: 14
Royalties (loc currency): 4782.93. Units: 1730180
Reading sheet: Apple & iTunes, Rows: 350, Columns: 17
Royalties (loc currency): 11.53870820121958. Units: 3128
Reading sheet: Spotify, Rows: 720, Columns: 14
Royalties (loc currency): 39.058739. Units: 25619
Reading sheet: Youtube, Rows: 188, Columns: 19
Royalties (loc currency): 12.896

/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43698/1008387074.py:568: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_final, df], ignore_index=True)
/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43698/1008387074.py:568: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_final, df], ignore_index=True)
/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43698/1008387074.py:568: FutureWarning: The behavior of DataFrame con

Reading sheet: 总表, Rows: 47, Columns: 16
Reading sheet: Apple & iTunes, Rows: 701, Columns: 17
Royalties (loc currency): 29.34336241087749. Units: 8024
Reading sheet: Spotify, Rows: 1304, Columns: 13
Royalties (loc currency): 60.129985. Units: 39241
Reading sheet: Youtube, Rows: 224, Columns: 19
Royalties (loc currency): 19.072438226663003. Units: 5894
Reading sheet: Facebook & Instagram, Rows: 195, Columns: 19
Royalties (loc currency): 0.785022086137. Units: 15824
Reading sheet: Amazon, Rows: 31, Columns: 19
Royalties (loc currency): 1.6377213995320001. Units: 331
Reading sheet: UMA, Rows: 3, Columns: 18
Royalties (loc currency): 0.00053862915. Units: 7
Reading sheet: Qobuz Stream, Rows: 2, Columns: 18
Royalties (loc currency): 0.09752140331800001. Units: 16
Reading sheet: TikTok, Rows: 4, Columns: 19
Royalties (loc currency): 0.005113353834999999. Units: 4
Reading sheet: Deezer, Rows: 17, Columns: 19
Royalties (loc currency): 0.34763781229999996. Units: 73
Reading sheet: Tidal, Rows:

/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43698/1008387074.py:568: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_final, df], ignore_index=True)
/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43698/1008387074.py:568: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_final, df], ignore_index=True)
/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43698/1008387074.py:568: FutureWarning: The behavior of DataFrame con

All DataFrames concatenated: Rows: 132037, Columns: 46
Merged DataFrame: Rows: 132037, Columns: 49
Royalties in USD: 39153.700087432764. Total units: 30001070

../../50 KM Group/Royalties/Statements/Karen/Earth/2025 Q4/as sent by Earth UNKNOWN FILE


In [6]:
df_fuyao, df_back_catalogue = align_columns(df_fuyao, df_back_catalogue)
df_x = pd.concat([df_fuyao, df_back_catalogue], ignore_index=True)

df_x, df_ziqia = align_columns(df_x, df_ziqia)
df_final = pd.concat([df_x, df_ziqia], ignore_index=True)

df_fx=pd.read_csv(fx_file)

usd_cny = df_fx.loc[df_fx['Currency']  == 'CNY', 'FX Rate'].values[0]

df_final["Statement Quarter"]=quarter
df_final["Royalties (CNY)"]= df_final["Royalties (USD)"]*usd_cny
df_final["Share MABB (CNY)"]= df_final["Royalties (CNY)"]*0.7
df_final["Share MABB (USD)"]= df_final["Royalties (USD)"]*0.7
df_final = df_final.drop(columns=['period_dt','客户付款货币','机械费','Label'],errors = 'ignore')
df_final = df_final.loc[:, ~df_final.columns.str.contains("^Unnamed")]
df_final[['Units','Royalties (USD)','Royalties (CNY)']]=df_final[['Units','Royalties (USD)','Royalties (CNY)']].fillna(0)

df_final = df_final.sort_index(axis=1)

print(f"Final DataFrame: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}")
for column in df_final.columns:        
        print(column)
print(f"Total fee in USD: {df_final['Royalties (USD)'].sum()}. Total units: {df_final['Units'].sum()}")

df_final.to_csv(path, index=False)


/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43698/1440251367.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_x = pd.concat([df_fuyao, df_back_catalogue], ignore_index=True)
/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43698/1440251367.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_x, df_ziqia], ignore_index=True)
/var/folders/r4/jcfnn_jn7d78r15gxq27mq2m0000gn/T/ipykernel_43698/1440251367.py:17: FutureWarning: Downcasting object dt

Final DataFrame: Rows: 135919, Columns: 48
Album
Apple Identifier
Artist
Composer Name
Content
Country
Currency
FX Rate
Gross Amount
ISRC
Incentivized Ads - Monthly Share of Revenue
Incentivized Ads - Total Plays
Issuance
Noise Content
Non-incentivized Ads - Monthly Share of Revenue
Non-incentivized Ads - Total Downloads
Non-incentivized Ads - Total Plays
Period
Platform
Play Type
Product
Product Type Identifier
Revenue
Royalties
Royalties (CNY)
Royalties (USD)
Sales Month
Sales Quarter
Sales or Return
Sales price
Share
Share Lyricist
Share MABB (CNY)
Share MABB (USD)
Share Master Owner
Share Performer
Share composer
Song
Song Length
Statement Quarter
Streaming category
Streaming type
Total Downloads
UPC
Units
Withholding Tax
税前金额 TWD
预扣税TWD
Total fee in USD: 40106.04962725825. Total units: 31877005
